# Core 04 - Results and Lineage

Objetivo: producir RunResult reales y derivar vistas, composicion y lineage sin
fabricar envelopes ni mutar evidencia.


## Parametros de la demostracion

| Parametro | Default | Proposito |
|---|---|---|
| symbols | RunResult y LineageMemory | Generar dos observaciones reales. |
| views | human, output, summary, view | Proyectar una sola fuente de verdad. |
| lineage | RunResult.lineage | Conservar procedencia y contexto compacto. |


## 1) Producir dos resultados reales


In [ ]:
from dataclasses import dataclass

import agentic_systems as toolkit

runtime = toolkit.runtime(provider="python-runtime")
system = toolkit.system(runtime=runtime)

@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.__all__,
        "package_version": toolkit.__version__,
    }

agent = system.agent(
    name="results_lineage_inspector",
    instructions="Ejecuta inspect_public_api y conserva la evidencia observada.",
    tools=[inspect_public_api],
    runtime=runtime,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
    policy=toolkit.RunPolicy(max_tool_calls=1, max_turns=2, temperature=0.0),
)
first_result = agent.run({"tool": "inspect_public_api", "input": {"symbol": "RunResult"}}, mode="eval")
second_result = agent.run({"tool": "inspect_public_api", "input": {"symbol": "LineageMemory"}}, mode="eval")
assert first_result.ok and second_result.ok


## 2) Proyectar el mismo resultado


In [ ]:
toolkit.human_result(first_result, title="Human RunResult", show_lineage=True)
toolkit.show_json(toolkit.run_result_output(first_result), title="run_result_output")
toolkit.show_json(toolkit.run_result_summary(first_result), title="run_result_summary")
toolkit.show_json(toolkit.run_result_view(first_result), title="run_result_view")

@dataclass
class ReviewNote:
    symbol: str
    accepted: bool

note = ReviewNote(symbol=first_result.data["symbol"], accepted=bool(first_result.data["is_public"]))
toolkit.show_json(note, title="Dataclass derivada del RunResult")


## 3) Derivar lineage y contexto compacto


In [ ]:
first_lineage = first_result.lineage(
    name="tutorial.results.first",
    question="RunResult pertenece a la API publica?",
    goal="Explicar evidencia de una ejecucion.",
)
toolkit.show(first_lineage, title="First Lineage")
toolkit.show_json(
    {"prompt_context": first_lineage.to_prompt_context(max_chars=900)},
    title="Compact prompt context",
)


## 4) Componer sin mutar


In [ ]:
combined = toolkit.compose_result(
    text="Se inspeccionaron dos simbolos de la API publica.",
    data={"observations": [toolkit.agent_output(first_result), toolkit.agent_output(second_result)]},
    results=[first_result, second_result],
    mode="lineage",
    input={"symbols": ["RunResult", "LineageMemory"]},
)
combined_lineage = combined.lineage(
    name="tutorial.results.combined",
    question="Que evidencia produjeron las dos ejecuciones?",
    goal="Conservar lineage compuesto sin mutaciones.",
)
toolkit.human_result(combined, title="Composed RunResult", show_lineage=True)
toolkit.show(combined_lineage, title="Combined Lineage")


## 5) API realmente ejercitada


In [ ]:
api_coverage = [
    "toolkit.runtime", "toolkit.system", "toolkit.tool", "system.agent", "agent.run",
    "toolkit.AgentContract", "toolkit.RunPolicy", "toolkit.human_result",
    "toolkit.run_result_output", "toolkit.run_result_summary", "toolkit.run_result_view",
    "toolkit.show_json", "toolkit.show", "RunResult.lineage",
    "LineageMemory.to_prompt_context", "toolkit.agent_output", "toolkit.compose_result",
]
toolkit.show_json(api_coverage, title="Results and lineage API coverage")


## Resultado esperado

Dos ejecuciones reales alimentan vistas, composicion y lineage trazable hasta la Tool.
